# Experiment 3: CPU Core Pressure Benchmark

This experiment evaluates the performance of the OCR + Mistral inference
pipeline under different CPU core limits.

- Electron / Flask UI is intentionally skipped
- Only the core pipeline is benchmarked
- One Docker run = one CPU limit
- Output: CSV file per CPU configuration
CPU Scaling & Latency Benchmarking

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


Import Required Modules

In [2]:
import os
import time
import random
import pandas as pd

from model.model_implement import run_pipeline_for_experiment

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Import succeeded!


## Read CPU Limit from Environment

The CPU limit is passed when launching the Docker container.
This value is recorded in the results for analysis.


In [3]:
cpu_cores = os.environ.get("CPU_CORES", "1.0")
print(f"Running experiment with CPU limit = {cpu_cores} cores")

Running experiment with CPU limit = 16.0 cores


## Dataset Sampling

We randomly sample a fixed number of PDFs to ensure consistency
across different CPU configurations.
Dataset Sampling

In [4]:
REPORTS_DIR = "../report/reports"

pdf_files = [
    os.path.join(root, f)
    for root, _, files in os.walk(REPORTS_DIR)
    for f in files if f.endswith(".pdf")
]

print(f"Total PDFs found: {len(pdf_files)}")

random.seed(42)
SELECTED_PDFS = random.sample(pdf_files, min(10, len(pdf_files)))


Total PDFs found: 168


## Experiment Runner

Each PDF is processed independently.
Latency and success/failure status are recorded.


In [5]:
def run_cpu_capped_experiment(cpu_limit, pdf_batch):
    results = []

    for pdf in pdf_batch:
        start_time = time.time()
        status = "success"
        error_type = None

        try:
            result = run_pipeline_for_experiment(
                pdf_path=pdf,
                language="English",
                model="mistral"
            )
            latency = time.time() - start_time

            if result is None or "error" in result:
                status = "failure"
                error_type = result.get("error") if result else "Unknown error"

        except Exception as e:
            latency = time.time() - start_time
            status = "failure"
            error_type = str(e)

        results.append({
            "pdf_name": os.path.basename(pdf),
            "cpu_cores": float(cpu_limit),
            "status": status,
            "latency_sec": latency,
            "error_type": error_type
        })

    return results


## Run Experiment and Save Results


In [6]:
results = run_cpu_capped_experiment(cpu_cores, SELECTED_PDFS)

df = pd.DataFrame(results)
output_file = f"cpu_pressure_results_{cpu_cores}CORES.csv"

df.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")


📄 Processing PDF: ../report/reports/Spanish/Spanish_LIVER_abnormal_4.pdf (1 pages)


🔍 Scanning 1 pages...


/app/src/pdf_processor.py:121: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral")


LLM Attempt 1...
Attempt 1 Failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
LLM Attempt 2...
Attempt 2 Failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
LLM Attempt 3...
Attempt 3 Failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))


📄 Processing PDF: ../report/reports/English/English_BLOOD_normal_1.pdf (1 pages)
🔍 Scanning 1 pages...


...

📄 Processing PDF: ../report/reports/Marathi/Marathi_LIVER_abnormal_4.pdf (1 pages)
🔍 Scanning 1 pages...


LLM Attempt 1...
Attempt 1 Failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
LLM Attempt 2...
Attempt 2 Failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
LLM Attempt 3...
Attempt 3 Failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))


📄 Processing PDF: ../report/reports/Bengali/Bengali_LIVER_normal_3.pdf (1 pages)
🔍 Scanning 1 pages...


LLM Attempt 1...
Attempt 1 Failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
LLM Attempt 2...
Attempt 2 Failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
LLM Attempt 3...
Attempt 3 Failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))


Results saved to cpu_pressure_results_16.0CORES.csv
